In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

# Ingest downloads new files to latest/, moving old files to archive/
# Spark file stream tracks processed files — only new files are appended
VOLUME_PATH = "/Volumes/gharchive_dev/v2_pipeline/files/latest/"

GHARCHIVE_SCHEMA = StructType([
    StructField("id", StringType()),
    StructField("type", StringType()),
    StructField("actor", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
        StructField("display_login", StringType()),
        StructField("gravatar_id", StringType()),
        StructField("url", StringType()),
        StructField("avatar_url", StringType()),
    ])),
    StructField("repo", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
        StructField("url", StringType()),
    ])),
    StructField("payload", StringType()),
    StructField("public", BooleanType()),
    StructField("created_at", StringType()),
    StructField("org", StructType([
        StructField("id", LongType()),
        StructField("login", StringType()),
        StructField("gravatar_id", StringType()),
        StructField("url", StringType()),
        StructField("avatar_url", StringType()),
    ])),
])


@dp.table(
    name="gharchive_bronze",
    comment="Raw GitHub Archive events - incrementally ingested from latest/ folder"
)
def gharchive_bronze():
    return (
        spark.readStream
        .format("json")
        .schema(GHARCHIVE_SCHEMA)
        .option("multiLine", "false")
        .load(VOLUME_PATH)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )